In [3]:
import pandas as pd
from datasets import load_dataset

!pip install datasets

#load the ag_news from hugging face
dataset = load_dataset("wangrongsheng/ag_news")

#convert the training split to a pandas Dataframe for easy preprocessing
df=pd.DataFrame(dataset['train'])
df.head()

,text,label
0,Wall St. Bears Claw Back Into the Black (Reute...,2
1,Carlyle Looks Toward Commercial Aerospace (Reu...,2
2,Oil and Economy Cloud Stocks' Outlook (Reuters...,2
3,Iraq Halts Oil Exports from Main Southern Pipe...,2
4,"Oil prices soar to all-time record, posing new...",2


In [8]:
train_df = pd.DataFrame(dataset["train"])
test_df = pd.DataFrame(dataset["test"])

In [9]:
import re

In [10]:
def clean_text(text):
    # Convert text to lowercase
    text = text.lower()

    # Remove punctuation, numbers, and special characters
    text = re.sub(r'[^a-z\s]', '', text)

    return text

In [11]:
train_df["text"] = train_df["text"].apply(clean_text)
test_df["text"] = test_df["text"].apply(clean_text)

In [12]:
train_df.head()

,text,label
0,wall st bears claw back into the black reuters...,2
1,carlyle looks toward commercial aerospace reut...,2
2,oil and economy cloud stocks outlook reuters r...,2
3,iraq halts oil exports from main southern pipe...,2
4,oil prices soar to alltime record posing new m...,2


In [13]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [14]:
#This creates a tokenizer object that will build a vocabulary.
tokenizer = Tokenizer()

In [15]:
#This scans all the training articles and assigns a unique integer ID to each unique word.
tokenizer.fit_on_texts(train_df["text"])

In [16]:
#Each article is converted from words into a sequence of numbers.
X_train = tokenizer.texts_to_sequences(train_df["text"])
X_test = tokenizer.texts_to_sequences(test_df["text"])

In [17]:
#Print the first 10 words in the vocabulary
print(list(tokenizer.word_index.items())[:10])

[('the', 1), ('to', 2), ('a', 3), ('of', 4), ('in', 5), ('and', 6), ('on', 7), ('for', 8), ('s', 9), ('that', 10)]


In [19]:
#To View the Tokenized Text
print(train_df["text"][0])
print(X_train[0])

wall st bears claw back into the black reuters reuters  shortsellers wall streets dwindlingband of ultracynics are seeing green again
[391, 324, 1525, 14260, 99, 54, 1, 812, 23, 23, 38863, 391, 1988, 50537, 4, 38864, 34, 3893, 737, 295]


In [20]:
#Import pad_sequences
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [21]:
#Set the Maximum Sequence Length
max_length = 50

In [25]:
#Pad and Truncate the Training Data
X_train = pad_sequences(
    X_train,
    maxlen=max_length,
    padding='post',
    truncating='post'
)

# Pad and Truncate the Testing Data
X_test = pad_sequences(
    X_test,
    maxlen=max_length,
    padding='post',
    truncating='post'
)

In [26]:
#Check the Shape
print("Training Data Shape:", X_train.shape)
print("Testing Data Shape:", X_test.shape)

Training Data Shape: (120000, 50)
Testing Data Shape: (7600, 50)


In [27]:
#View a Padded Sequence
print(X_train[0])

[  391   324  1525 14260    99    54     1   812    23    23 38863   391
  1988 50537     4 38864    34  3893   737   295     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0]


In [28]:
from tensorflow.keras.utils import to_categorical

In [29]:
y_train = to_categorical(train_df["label"])
y_test = to_categorical(test_df["label"])

In [30]:
print(train_df["label"].head())

0    2
1    2
2    2
3    2
4    2
Name: label, dtype: int64


In [31]:
print(y_train[:5])

[[0. 0. 1. 0.]
 [0. 0. 1. 0.]
 [0. 0. 1. 0.]
 [0. 0. 1. 0.]
 [0. 0. 1. 0.]]


In [32]:
#Check the Shape
print("Training Labels Shape:", y_train.shape)
print("Testing Labels Shape:", y_test.shape)

Training Labels Shape: (120000, 4)
Testing Labels Shape: (7600, 4)


In [33]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

In [34]:
vocab_size = len(tokenizer.word_index) + 1
print("Vocabulary Size:", vocab_size)

Vocabulary Size: 91344


In [35]:
model = Sequential()

# Embedding Layer
model.add(Embedding(input_dim=vocab_size,
                    output_dim=64,
                    input_length=max_length))

# SimpleRNN Layer
model.add(SimpleRNN(64))

# Output Layer
model.add(Dense(4, activation='softmax'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [36]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [37]:
# Compile the RNN model
model.compile(
    optimizer='adam',                  # Optimizer that updates model weights
    loss='categorical_crossentropy',   # Loss function for multi-class classification
    metrics=['accuracy']               # Evaluate the model using accuracy
)

In [38]:
# Train the RNN model
history = model.fit(
    X_train,                # Training input data
    y_train,                # One-hot encoded training labels
    epochs=5,               # Number of times the model sees the entire training dataset
    batch_size=64,          # Number of samples processed before updating the weights
    validation_split=0.2    # Use 20% of the training data for validation
)

Epoch 1/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 127s 83ms/step - accuracy: 0.8234 - loss: 0.5150 - val_accuracy: 0.8481 - val_loss: 0.4524
Epoch 2/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 128s 85ms/step - accuracy: 0.9146 - loss: 0.2751 - val_accuracy: 0.8650 - val_loss: 0.4259
Epoch 3/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 141s 85ms/step - accuracy: 0.9407 - loss: 0.1910 - val_accuracy: 0.8661 - val_loss: 0.4497
Epoch 4/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 140s 83ms/step - accuracy: 0.9548 - loss: 0.1452 - val_accuracy: 0.8671 - val_loss: 0.4370
Epoch 5/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 141s 83ms/step - accuracy: 0.9648 - loss: 0.1143 - val_accuracy: 0.8598 - val_loss: 0.5092


In [39]:
model.compile(
    optimizer='adam',                  # Optimizer used to update weights
    loss='categorical_crossentropy',   # Loss function for multi-class classification
    metrics=['accuracy']               # Display accuracy during training
)
# Train the RNN Model

history = model.fit(
    X_train,                # Input training data
    y_train,                # Target labels
    epochs=5,               # Number of training iterations
    batch_size=64,          # Samples processed in each batch
    validation_split=0.2    # Reserve 20% of training data for validation
)

Epoch 1/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 128s 84ms/step - accuracy: 0.9698 - loss: 0.0975 - val_accuracy: 0.8612 - val_loss: 0.5181
Epoch 2/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 125s 83ms/step - accuracy: 0.9777 - loss: 0.0730 - val_accuracy: 0.8102 - val_loss: 0.7196
Epoch 3/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 125s 84ms/step - accuracy: 0.9796 - loss: 0.0682 - val_accuracy: 0.8454 - val_loss: 0.6418
Epoch 4/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 125s 83ms/step - accuracy: 0.9827 - loss: 0.0555 - val_accuracy: 0.8635 - val_loss: 0.5442
Epoch 5/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 126s 84ms/step - accuracy: 0.9830 - loss: 0.0557 - val_accuracy: 0.8361 - val_loss: 0.6565


In [40]:
# Evaluate the trained model on the test dataset
loss, accuracy = model.evaluate(
    X_test,     # Test input data
    y_test      # Test labels
)

# Display the evaluation results
print("Test Loss:", loss)
print("Test Accuracy:", accuracy)

238/238 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.8630 - loss: 0.5622
Test Loss: 0.562201976776123
Test Accuracy: 0.8630263209342957


In [41]:
# Display the training accuracy from the last epoch
print("Training Accuracy:", history.history['accuracy'][-1])

# Display the validation accuracy from the last epoch
print("Validation Accuracy:", history.history['val_accuracy'][-1])

Training Accuracy: 0.9830312728881836
Validation Accuracy: 0.8360833525657654


In [42]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (64, 50, 64)           │     5,846,016 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (64, 64)               │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (64, 4)                │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 17,563,598 (67.00 MB)

 Trainable params: 5,854,532 (22.33 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 11,709,066 (44.67 MB)